## Parameter für Optimierungslauf

In [0]:
%sh

cat > /tmp/pypsa_run_config.sh <<'EOF'

# ==========================================
# HIER EINSTELLEN
# ==========================================

YEAR="2045"

# Entspricht deiner config:
RUN_PREFIX="20261023_"
SCENARIO="SR27_S3"

# Name, unter dem diese Optimierung gespeichert wird
SAVE_NAME="original"

# Speicherort
SAVE_DIR="/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results"

EOF

source /tmp/pypsa_run_config.sh

echo "=========================================="
echo "Einstellungen"
echo "=========================================="
echo "Jahr:         $YEAR"
echo "Run Prefix:   $RUN_PREFIX"
echo "Szenario:     $SCENARIO"
echo "Speichername: $SAVE_NAME"
echo "Speicherort:  $SAVE_DIR"
echo "=========================================="

## PyPSA-DE vorbereiten

In [0]:
%sh

set -euo pipefail

source /tmp/pypsa_run_config.sh

export PATH="$HOME/.pixi/bin:$PATH"

if ! command -v pixi >/dev/null 2>&1; then
  curl -fsSL https://pixi.sh/install.sh | sh
  export PATH="$HOME/.pixi/bin:$PATH"
fi

SRC="/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test/"
DST="/local_disk0/pypsa-de-test/"

mkdir -p "$DST"

rsync -a \
  --exclude=".pixi/" \
  --exclude="scripts_LuKa/05_Optimize_SR27-3" \
  "$SRC" "$DST"

cd "$DST"

pixi install

pixi run snakemake --version

PYTHONPATH="" pixi run snakemake -c1 build_scenarios


## Gewähltes Jahr optimieren und separat sichern

In [0]:
%sh

set -euo pipefail

source /tmp/pypsa_run_config.sh

export PATH="$HOME/.pixi/bin:$PATH"

cd /local_disk0/pypsa-de-test


# Pfade zusammensetzen
RESULTS="results/${RUN_PREFIX}/${SCENARIO}/networks"

TARGET="${RESULTS}/base_s_27__none_${YEAR}.nc"

BACKUP="${SAVE_DIR}/${SAVE_NAME}"

BACKUP_FILE="${BACKUP}/$(basename "$TARGET")"


# Einstellungen anzeigen
echo ""
echo "=========================================="
echo "OPTIMIERUNGSLAUF"
echo "=========================================="
echo "Jahr:         $YEAR"
echo "Run Prefix:   $RUN_PREFIX"
echo "Szenario:     $SCENARIO"
echo ""
echo "Target:"
echo "$TARGET"
echo ""
echo "Backup:"
echo "$BACKUP_FILE"
echo "=========================================="
echo ""


# Backup-Ordner anlegen
mkdir -p "$BACKUP"


# Schutz vor Überschreiben
if [ -e "$BACKUP_FILE" ]; then
  echo ""
  echo "FEHLER:"
  echo "Die Datei existiert bereits:"
  echo ""
  echo "$BACKUP_FILE"
  echo ""
  echo "Bitte in Zelle 1 einen anderen SAVE_NAME wählen."
  exit 1
fi

# Optimierung
echo "=== Optimierung $YEAR ==="

PYTHONPATH="" pixi run snakemake -c8 --jobs 1 \
  "$TARGET" \
  --resources mem_mb=60000 \
  --set-resources solve_sector_network_myopic:mem_mb=60000


# Prüfen, ob Ergebnis erzeugt wurde
if [ ! -f "$TARGET" ]; then
  echo "FEHLER: Ergebnis wurde nicht gefunden:"
  echo "$TARGET"
  exit 1
fi


# Ergebnis sichern
echo ""
echo "=== Ergebnis sichern ==="

cp "$TARGET" "$BACKUP_FILE"


# Kopie überprüfen
cmp "$TARGET" "$BACKUP_FILE"


echo ""
echo "=========================================="
echo "ERFOLGREICH"
echo "=========================================="
echo "Jahr:      $YEAR"
echo "Szenario:  $SCENARIO"
echo ""
echo "Gespeichert unter:"
echo "$BACKUP_FILE"
echo "=========================================="